# Vertex box exploration

The neutrino vertex is always at **time ≈ 0** (top of the image, row → 0)
and **wire ≈ 250** (centre of the wire axis, col ≈ 250).

Strategy:
1. Define a **search region** around (0, 250) — a coarse window that is guaranteed to contain the vertex.
2. Inside the search region, find the **local density maximum**: the pixel with the most active neighbours within radius *r*.
3. Place the final **evaluation box** centred on that refined vertex.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
%matplotlib inline

# ── Change this path to your .npz ──────────────────────────────────────────
NPZ_PATH = Path("../../dino_checkpoints/test_cropping2/features_ep10.npz")
# ───────────────────────────────────────────────────────────────────────────

data      = np.load(NPZ_PATH)
positions = data["positions"]   # [N_valid, 2]  (row, col)
charges   = data["charges"]     # [N_valid, 1]  raw pixel charge (ADC)
offsets   = data["offsets"]     # [N_images+1]
labels    = data["labels"]      # [N_images]

CLASS_NAMES = ["numu", "nue", "nutau", "NC"]
print(f"Images: {len(labels)}   Valid pixels: {positions.shape[0]}")
print({ CLASS_NAMES[c]: int((labels==c).sum()) for c in np.unique(labels) })

## Parameters

In [ ]:
# ── Coarse search region around the known vertex neighbourhood ──────────────
SEARCH_ROW  = 0     # search rows [SEARCH_ROW, SEARCH_ROW + SEARCH_H)
SEARCH_COL  = 250   # search cols [SEARCH_COL - SEARCH_W, SEARCH_COL + SEARCH_W)
SEARCH_H    = 100    # how far down from row 0 to look
SEARCH_W    = 100   # half-width around col 250

# ── Charge threshold ────────────────────────────────────────────────────────
CHARGE_THRESHOLD = 50.0   # minimum ADC value; pixels below this are ignored

# ── Option A: earliest pixel with minimum local support ─────────────────────
NEIGHBOR_R    = 3.0  # radius (pixels) for neighbour count
MIN_NEIGHBORS = 15    # minimum neighbours required to not be a blip

# ── Option B: row projection ─────────────────────────────────────────────────
MIN_HITS_ROW  = 50    # minimum above-threshold pixels in a row to be considered

# ── Local density radius (pixels) — used by original find_vertex only ───────
DENSITY_R   = 100.0

# ── Final evaluation box (centred on the refined vertex) ───────────────────
BOX_H = 50   # half-height
BOX_W = 30   # half-width

## Vertex finder

In [ ]:
def find_vertex(pos: np.ndarray, chg: np.ndarray,
                search_row: int, search_col: int,
                search_h: int, search_w: int,
                density_r: float, charge_threshold: float) -> tuple[float, float]:
    """
    1. Restrict to active pixels inside the search region.
    2. Among those, return the one with the highest count of active neighbours
       (from the full image) within `density_r` pixels.
    Falls back to the search-region centroid if no pixel is found.
    """
    rows, cols = pos[:, 0], pos[:, 1]
    chg_vals   = chg[:, 0] if chg.ndim == 2 else chg

    in_search = (
        (rows >= search_row) & (rows < search_row + search_h) &
        (cols >= search_col - search_w) & (cols < search_col + search_w)
        & (chg_vals >= charge_threshold)
    )
    candidates = pos[in_search]    # pixels inside the search region

    if len(candidates) == 0:
        return float(search_row), float(search_col)   # fallback

    # Count neighbours in the *full* image within radius density_r
    # candidates: [C, 2],  pos: [N, 2]
    diff   = candidates[:, None, :] - pos[None, :, :]   # [C, N, 2]
    counts = (np.linalg.norm(diff, axis=-1) < density_r).sum(axis=1)  # [C]

    best = candidates[counts.argmax()]
    return float(best[0]), float(best[1])

In [ ]:
def find_vertex_min_time(pos: np.ndarray,
                         chg: np.ndarray,
                         search_row: int, search_col: int,
                         search_h: int, search_w: int,
                         charge_threshold: float = 0.0) -> tuple[float, float]:
    """
    1. Restrict to active pixels inside the search region above charge_threshold.
    2. Among those, return the one with the smallest row (earliest time).
    Falls back to the search-region centroid if no pixel passes the cuts.
    """
    rows, cols = pos[:, 0], pos[:, 1]
    chg_vals   = chg[:, 0] if chg.ndim == 2 else chg

    in_search = (
        (rows >= search_row) & (rows < search_row + search_h) &
        (cols >= search_col - search_w) & (cols < search_col + search_w) &
        (chg_vals >= charge_threshold)
    )
    candidates = pos[in_search]

    if len(candidates) == 0:
        return float(search_row), float(search_col)   # fallback

    best = candidates[candidates[:, 0].argmin()]
    return float(best[0]), float(best[1])

In [ ]:
def find_vertex_earliest_connected(pos: np.ndarray,
                                   chg: np.ndarray,
                                   search_row: int, search_col: int,
                                   search_h: int, search_w: int,
                                   charge_threshold: float = 0.0,
                                   neighbor_r: float = 5.0,
                                   min_neighbors: int = 3) -> tuple[float, float]:
    """
    Option A — earliest pixel with minimum local support.

    1. Restrict to above-threshold pixels inside the search region.
    2. For each candidate, count how many *other candidates* (within the search
       region) lie within `neighbor_r` pixels — using only the search-region
       pixels keeps the cost manageable and avoids shower activity outside.
    3. Keep only candidates with >= `min_neighbors` neighbours (filters blips).
    4. Among the survivors, return the one with the smallest row (earliest time).
    Falls back to the search-region centroid if no pixel passes all cuts.
    """
    rows, cols = pos[:, 0], pos[:, 1]
    chg_vals   = chg[:, 0] if chg.ndim == 2 else chg

    in_search = (
        (rows >= search_row) & (rows < search_row + search_h) &
        (cols >= search_col - search_w) & (cols < search_col + search_w) &
        (chg_vals >= charge_threshold)
    )
    candidates = pos[in_search]   # [C, 2]

    if len(candidates) == 0:
        return float(search_row), float(search_col)

    # Neighbour counts within the search region only
    diff   = candidates[:, None, :] - candidates[None, :, :]   # [C, C, 2]
    # Exclude self by subtracting 1
    counts = (np.linalg.norm(diff, axis=-1) < neighbor_r).sum(axis=1) - 1  # [C]

    connected = candidates[counts >= min_neighbors]
    if len(connected) == 0:
        # Fallback: relax to the single best-connected pixel
        connected = candidates[[counts.argmax()]]

    best = connected[connected[:, 0].argmin()]
    return float(best[0]), float(best[1])

In [ ]:
def find_vertex_row_projection(pos: np.ndarray,
                               chg: np.ndarray,
                               search_row: int, search_col: int,
                               search_h: int, search_w: int,
                               charge_threshold: float = 0.0,
                               min_hits_row: int = 3) -> tuple[float, float]:
    """
    Option B — row projection, first active row.

    1. Restrict to above-threshold pixels inside the search region.
    2. Walk rows from search_row downward; return the first row that contains
       >= `min_hits_row` such pixels.
    3. The column is the charge-weighted centroid of that row's pixels.
    Falls back to the search-region centroid if no qualifying row is found.
    """
    rows, cols = pos[:, 0], pos[:, 1]
    chg_vals   = chg[:, 0] if chg.ndim == 2 else chg

    in_search = (
        (rows >= search_row) & (rows < search_row + search_h) &
        (cols >= search_col - search_w) & (cols < search_col + search_w) &
        (chg_vals >= charge_threshold)
    )
    s_rows = rows[in_search].astype(int)
    s_cols = cols[in_search]
    s_chg  = chg_vals[in_search]

    if len(s_rows) == 0:
        return float(search_row), float(search_col)

    for r in range(search_row, search_row + search_h):
        mask = (s_rows == r)
        if mask.sum() >= min_hits_row:
            w = s_chg[mask]
            vc = float(np.average(s_cols[mask], weights=w))
            return float(r), vc

    # Fallback: no row met the threshold — return earliest pixel
    best_idx = s_rows.argmin()
    return float(s_rows[best_idx]), float(s_cols[best_idx])

## Visual check — fixed search region vs refined vertex

In [ ]:
METHODS = [
    ("density",    "purple", lambda p, c: find_vertex(
        p, c, SEARCH_ROW, SEARCH_COL, SEARCH_H, SEARCH_W, DENSITY_R, CHARGE_THRESHOLD)),
    ("min_time",   "C0",     lambda p, c: find_vertex_min_time(
        p, c, SEARCH_ROW, SEARCH_COL, SEARCH_H, SEARCH_W, CHARGE_THRESHOLD)),
    ("opt-A\nearliest\nconnected", "C2", lambda p, c: find_vertex_earliest_connected(
        p, c, SEARCH_ROW, SEARCH_COL, SEARCH_H, SEARCH_W, CHARGE_THRESHOLD, NEIGHBOR_R, MIN_NEIGHBORS)),
    ("opt-B\nrow\nprojection",    "C3", lambda p, c: find_vertex_row_projection(
        p, c, SEARCH_ROW, SEARCH_COL, SEARCH_H, SEARCH_W, CHARGE_THRESHOLD, MIN_HITS_ROW)),
]

N_SHOW_CMP = 4
rng_cmp    = np.random.default_rng(0)

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    cls_imgs = np.where(labels == cls_idx)[0]
    if len(cls_imgs) == 0:
        continue
    chosen = rng_cmp.choice(cls_imgs, size=min(N_SHOW_CMP, len(cls_imgs)), replace=False)

    n_methods = len(METHODS)
    fig, axes = plt.subplots(len(chosen), n_methods,
                             figsize=(4 * n_methods, 4 * len(chosen)),
                             squeeze=False)

    for row_i, img_idx in enumerate(chosen):
        sl   = slice(offsets[img_idx], offsets[img_idx + 1])
        pos  = positions[sl].astype(np.float32)
        chg  = charges[sl]
        rows, cols = pos[:, 0], pos[:, 1]
        abve_thr = (chg[:, 0] >= CHARGE_THRESHOLD) if chg.ndim == 2 else (chg >= CHARGE_THRESHOLD)

        for col_i, (name, color, fn) in enumerate(METHODS):
            ax = axes[row_i][col_i]
            vr, vc = fn(pos, chg)

            in_box = (
                (rows >= vr - BOX_H) & (rows < vr + BOX_H) &
                (cols >= vc - BOX_W) & (cols < vc + BOX_W)
            )

            ax.scatter(cols, rows, s=0.3, c="black", linewidths=0)
            ax.scatter(cols[abve_thr], rows[abve_thr], s=0.3, c="C1", linewidths=0)
            ax.scatter(cols[in_box], rows[in_box], s=0.8, c="magenta", linewidths=0, zorder=3)

            # Search region
            ax.add_patch(patches.Rectangle(
                (SEARCH_COL - SEARCH_W, SEARCH_ROW), 2 * SEARCH_W, SEARCH_H,
                linewidth=1.0, edgecolor="C0", facecolor="none", linestyle="--", zorder=4,
            ))
            # Evaluation box
            ax.add_patch(patches.Rectangle(
                (vc - BOX_W, vr - BOX_H), 2 * BOX_W, 2 * BOX_H,
                linewidth=1.5, edgecolor=color, facecolor="none", zorder=5,
            ))
            ax.scatter([vc], [vr], c=color, marker="+", s=120, linewidths=1.5, zorder=6)

            ax.set_xlim(0, 500); ax.set_ylim(500, 0)
            ax.set_aspect("equal")

            title = f"img {img_idx}  ({vr:.0f},{vc:.0f})  n={in_box.sum()}"
            if row_i == 0:
                title = f"[{name}]\n" + title
            ax.set_title(title, fontsize=7)

    fig.suptitle(
        f"{cls_name}  |  search (dashed)  box color = method  thr={CHARGE_THRESHOLD}",
        fontsize=10,
    )
    plt.tight_layout()
    plt.show()

In [ ]:
N_SHOW = 2
rng    = np.random.default_rng(0)

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    cls_imgs = np.where(labels == cls_idx)[0]
    if len(cls_imgs) == 0:
        continue
    chosen = rng.choice(cls_imgs, size=min(N_SHOW, len(cls_imgs)), replace=False)

    fig, axes = plt.subplots(1, len(chosen), figsize=(5 * len(chosen), 5), squeeze=False)

    for ax, img_idx in zip(axes[0], chosen):
        sl   = slice(offsets[img_idx], offsets[img_idx + 1])
        pos  = positions[sl].astype(np.float32)
        chg  = charges[sl]
        rows, cols = pos[:, 0], pos[:, 1]

        # Refined vertex
        #vr, vc = find_vertex(pos, chg, SEARCH_ROW, SEARCH_COL, SEARCH_H, SEARCH_W, DENSITY_R, CHARGE_THRESHOLD)
        #vr, vc = find_vertex_min_time(pos, chg, SEARCH_ROW, SEARCH_COL, SEARCH_H, SEARCH_W, CHARGE_THRESHOLD)
        vr, vc = find_vertex_earliest_connected(pos, chg, SEARCH_ROW, SEARCH_COL, SEARCH_H, SEARCH_W, CHARGE_THRESHOLD, NEIGHBOR_R, MIN_NEIGHBORS)
        #vr, vc = find_vertex_row_projection(pos, chg, SEARCH_ROW, SEARCH_COL, SEARCH_H, SEARCH_W, CHARGE_THRESHOLD, MIN_HITS_ROW)

        # Pixels inside the final evaluation box
        in_box = (
            (rows >= vr - BOX_H) & (rows < vr + BOX_H) &
            (cols >= vc - BOX_W) & (cols < vc + BOX_W)
        )

        abve_thr = (chg[:, 0] >= CHARGE_THRESHOLD) if chg.ndim == 2 else (chg >= CHARGE_THRESHOLD)

        # Draw
        ax.scatter(cols, rows, s=0.5, c="black", linewidths=0)
        ax.scatter(cols[abve_thr], rows[abve_thr], s=0.5, c="C1", linewidths=0)
        ax.scatter(cols[in_box], rows[in_box], s=1.0, c="magenta", linewidths=0, zorder=3)

        # Search region (dashed blue)
        ax.add_patch(patches.Rectangle(
            (SEARCH_COL - SEARCH_W, SEARCH_ROW), 2 * SEARCH_W, SEARCH_H,
            linewidth=1.2, edgecolor="C0", facecolor="none",
            linestyle="--", zorder=4,
        ))
        # Evaluation box (solid red)
        ax.add_patch(patches.Rectangle(
            (vc - BOX_W, vr - BOX_H), 2 * BOX_W, 2 * BOX_H,
            linewidth=1.5, edgecolor="red", facecolor="none", zorder=5,
        ))
        # Vertex marker
        ax.scatter([vc], [vr], c="red", marker="+", s=100, zorder=6)

        #ax.set_facecolor("black")
        ax.set_xlim(0, 500); ax.set_ylim(500, 0)
        ax.set_aspect("equal")
        ax.set_title(
            f"[{cls_name}] img {img_idx}\n"
            f"vertex=({vr:.0f},{vc:.0f})  in_box={in_box.sum()}/{len(pos)}",
            fontsize=8,
        )

    fig.suptitle(
        f"{cls_name}  |  search (dashed blue)  eval box (red)  thr={CHARGE_THRESHOLD}",
        fontsize=10,
    )
    plt.tight_layout()
    plt.show()

## Pixel counts inside evaluation box per class

In [ ]:
N_EVAL = 300
rng2   = np.random.default_rng(1)

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    cls_imgs = np.where(labels == cls_idx)[0]
    if len(cls_imgs) == 0:
        continue
    sample = rng2.choice(cls_imgs, size=min(N_EVAL, len(cls_imgs)), replace=False)
    in_box = []
    for img_idx in sample:
        sl  = slice(offsets[img_idx], offsets[img_idx + 1])
        pos = positions[sl].astype(np.float32)
        chg = charges[sl]
        rows, cols = pos[:, 0], pos[:, 1]
        # vr, vc = find_vertex(pos, SEARCH_ROW, SEARCH_COL, SEARCH_H, SEARCH_W, DENSITY_R)
        vr, vc = find_vertex_min_time(pos, chg, SEARCH_ROW, SEARCH_COL,
                                      SEARCH_H, SEARCH_W, CHARGE_THRESHOLD)
        n = int((
            (rows >= vr - BOX_H) & (rows < vr + BOX_H) &
            (cols >= vc - BOX_W) & (cols < vc + BOX_W)
        ).sum())
        in_box.append(n)
    in_box = np.array(in_box)
    print(f"{cls_name:6s}  median={np.median(in_box):.0f}  mean={in_box.mean():.1f}  "
          f"min={in_box.min()}  max={in_box.max()}  zero_frac={(in_box==0).mean():.1%}")

## Chosen parameters → CLI flags

```bash
python -m dino.diagnostics.plot_knn features.npz \
    --vertex_knn \
    --search_row 0 --search_col 250 \
    --search_h SEARCH_H --search_w SEARCH_W \
    --density_r DENSITY_R \
    --box_h BOX_H --box_w BOX_W \
    --n_vertex_images 500
```